# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*   **Unit of Analysis (what one row means)**: A single page (`content_hash_id`) for a specific client (`client_hash_id`) aggregated over a monthly observation window. Every row in our feature set represents one page's monthly performance.
*   **Which table(s) we'll use**: We use the warehouse table `fact_content_daily_performance` (to build daily performance features and outcome labels), joined with `dim_content` (to fetch content metadata like age, word count, and type) and `dim_clients` (to check history availability).
*   **Which time window**: We use a 30-day feature/observation window (March 2026: `2026-03-01` to `2026-03-31`) to build historical features, and a subsequent 30-day prediction/outcome window (April 2026: `2026-04-01` to `2026-04-30`) to define the target label.
*   **What we predict or rank (label/proxy)**: We predict `is_declining` (a binary proxy indicating whether the organic impressions in the outcome window drop by >= 20% compared to the feature window).
*   **Deliberately Excluded**: We deliberately exclude `trend_pct` and `trend_direction` from our feature window (since they directly leak the label), and any search or analytics metrics from the future prediction window to avoid lookahead bias.

In [5]:
# Find repo root and load the dataset
import os, pandas as pd, numpy as np
for _ in range(5):
    if os.path.isdir("data/raw"): break
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working directory:", os.getcwd())
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique clients: {df['client_id'].nunique()} | Unique pages: {df['content_id'].nunique():,}")

Working directory: d:\Flyrank_internship\Ml_intership
Dataset shape: 30,000 rows x 44 columns
Unique clients: 32 | Unique pages: 30,000


## 2. Fields: feature / label / context / excluded

*   **Features (Safe to model)**:
    *   *Search performance features*: `impressions_30d`, `clicks_30d`, `avg_position_30d`.
    *   *User engagement features*: `engagement_rate_30d` (engaged sessions / sessions * 100).
    *   *Metadata features*: `content_age_at_decision` (creation age of the content item in days).
*   **Label / Proxy**:
    *   `is_declining` (derived from organic impressions dropping by >= 20% in the future 30-day window).
*   **Context (Keys & Helpers)**:
    *   `client_hash_id`, `content_hash_id` (used for grouping, joining, and cross-validation splits; never modeled directly).
*   **Excluded (Why)**:
    *   `trend_pct` / `trend_direction` in the observation window: direct leakage (the label is calculated from them).
    *   Post-decision performance metrics (e.g., impressions or clicks in the outcome window): not knowable at the moment of prediction.

In [6]:
# Verify that trend_pct is highly correlated with trend_direction and would leak the label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
mean_pct_by_trend = df.groupby("trend_direction")["trend_pct"].mean()
print("Mean trend_pct by trend direction:")
print(mean_pct_by_trend.to_string())
assert "trend_pct" not in ["content_id", "client_id"], "Self-check: trend_pct must be excluded from feature vectors!"

Mean trend_pct by trend direction:
trend_direction
down      -58.113830
flat             NaN
new              NaN
stable     -3.185944
up        190.673997


## 3. Verify it with queries (grain, counts, missing values, windows)

We verify our data contract and prove three facts with SQL queries on the warehouse dataset for `month=2026-03`:
1.  **Fact 1 (The Grain)**: Prove that the daily fact table's grain is exactly `report_date` × `client_hash_id` × `content_hash_id` (zero rows return when grouping by these and checking for counts > 1).
2.  **Fact 2 (Row Count & Date Span)**: Verify the total row count and exact date span for the mid-panel month (`month=2026-03`).
3.  **Fact 3 (Availability)**: Check how many rows survive when filtering with `ga4_data_available IS TRUE` (since early history is zero-filled).

We also build our **Five Features Frame** for March 2026 and run a **Leakage Experiment** to demonstrate the leakage trap (comparing a model trained with a leaked future column against an honest baseline model).

In [7]:
import os
import duckdb
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Hugging Face Access & Local Fallback Setup
token = os.environ.get("HF_TOKEN")
if not token and os.path.exists(".env"):
    with open(".env", "r") as f:
        for line in f:
            if line.startswith("HF_TOKEN="):
                token = line.strip().split("=", 1)[1]

use_mock = True
con = None
local_dir = None

if token:
    try:
        from huggingface_hub import hf_hub_download
        print("Checking/Downloading dataset files from Hugging Face...")
        files_to_download = [
            'dim_clients.parquet',
            'dim_content.parquet',
            'fact_content_daily_performance/month=2026-03/data_0.parquet',
            'fact_content_daily_performance/month=2026-04/data_0.parquet',
        ]
        local_paths = {}
        for filename in files_to_download:
            local_paths[filename] = hf_hub_download(
                repo_id='FlyRank/internship-warehouse',
                filename=filename,
                repo_type='dataset',
                token=token
            )
        local_dir = os.path.dirname(os.path.dirname(local_paths['fact_content_daily_performance/month=2026-03/data_0.parquet']))
        con = duckdb.connect()
        use_mock = False
        print("Hugging Face dataset connected successfully!")
    except Exception as e:
        print(f"HF access failed ({e}). Falling back to local mock simulation...")

if use_mock:
    print("Initializing in-memory mock database for warehouse simulation...")
    con = duckdb.connect()
    
    # Load local CSV
    local_df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
    local_df = local_df.rename(columns={
        'client_id': 'client_hash_id',
        'content_id': 'content_hash_id'
    })
    
    # Register dim_content
    dim_content = pd.DataFrame({
        'content_hash_id': local_df['content_hash_id'],
        'content_age_days': local_df['content_age_days'],
        'word_count': local_df['word_count'],
        'content_type': local_df['content_type']
    })
    con.register('dim_content', dim_content)
    
    # Register dim_clients
    dim_clients = pd.DataFrame({
        'client_hash_id': local_df['client_hash_id'].unique(),
        'access_profile': 'standard',
        'gsc_data_start': pd.to_datetime('2025-01-27'),
        'ga4_data_start': pd.to_datetime('2025-02-15')
    })
    con.register('dim_clients', dim_clients)
    
    # Register fact_daily
    sampled_pages = local_df.sample(n=min(3000, len(local_df)), random_state=42)
    dates = pd.date_range(start='2026-03-01', end='2026-04-30')
    
    rows = []
    np.random.seed(42)
    for _, page in sampled_pages.iterrows():
        avg_daily_imp = page['impressions_90d'] / 90.0
        avg_daily_clk = page['clicks_90d'] / 90.0
        avg_daily_sess = page['sessions_90d'] / 90.0
        avg_daily_esess = page['engaged_sessions_90d'] / 90.0
        
        ga4_avail = np.random.rand() > 0.1
        will_decline = np.random.rand() > 0.6
        
        for d in dates:
            decline_factor = 0.6 if (d.month == 4 and will_decline) else 1.0
            
            imp = max(0, int(np.random.normal(avg_daily_imp * decline_factor, max(1, avg_daily_imp * 0.1))))
            clk = max(0, int(np.random.normal(avg_daily_clk * decline_factor, max(0.5, avg_daily_clk * 0.1))))
            sess = max(0, int(np.random.normal(avg_daily_sess * decline_factor, max(0.5, avg_daily_sess * 0.1)))) if ga4_avail else 0
            esess = max(0, int(np.random.normal(avg_daily_esess * decline_factor, max(0.5, avg_daily_esess * 0.1)))) if ga4_avail else 0
            
            rows.append({
                'report_date': d,
                'client_hash_id': page['client_hash_id'],
                'content_hash_id': page['content_hash_id'],
                'gsc_impressions': imp,
                'gsc_clicks': clk,
                'gsc_avg_position': page['avg_position'],
                'ga4_sessions': sess,
                'ga4_engaged_sessions': esess,
                'ga4_data_available': ga4_avail,
                'gsc_data_available': True
            })
            
    fact_daily = pd.DataFrame(rows)
    con.register('fact_daily', fact_daily)
    print("Local mock database initialized successfully!")

def run_query(sql_query):
    cleaned_query = sql_query
    if use_mock:
        cleaned_query = re.sub(r"read_parquet\([^)]*dim_clients\.parquet[^)]*\)", "dim_clients", cleaned_query)
        cleaned_query = re.sub(r"read_parquet\([^)]*dim_content\.parquet[^)]*\)", "dim_content", cleaned_query)
        cleaned_query = re.sub(r"read_parquet\([^)]*fact_content_daily_performance/month=2026-03/\*\.parquet[^)]*\)", "fact_daily WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'", cleaned_query)
        cleaned_query = re.sub(r"read_parquet\([^)]*fact_content_daily_performance/\*\*/\*\.parquet[^)]*\)", "fact_daily", cleaned_query)
        cleaned_query = re.sub(r"read_parquet\([^)]*fact_content_daily_performance[^)]*\)", "fact_daily", cleaned_query)
    return con.execute(cleaned_query).df()

REL_PATH = 'hf://datasets/FlyRank/internship-warehouse'
if use_mock:
    TABLES = {
        'dim_clients': 'dim_clients',
        'dim_content': 'dim_content',
        'fact_daily': 'fact_daily'
    }
else:
    TABLES = {
        'dim_clients': f"read_parquet('{os.path.join(local_dir, 'dim_clients.parquet')}')",
        'dim_content': f"read_parquet('{os.path.join(local_dir, 'dim_content.parquet')}')",
        'fact_daily': f"read_parquet('{os.path.join(local_dir, 'fact_content_daily_performance', 'month=2026-03', '*.parquet')}')"
    }

print("==================================================")
print("TWO: VERIFICATION QUERIES (month=2026-03)")
print("==================================================")

print("\n--- Fact 1: Grain Test (Should return 0 rows if grain holds) ---")
q1 = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY 1, 2, 3
HAVING c > 1
LIMIT 5
"""
print(run_query(q1))

print("\n--- Fact 2: Row Count & Date Span ---")
q2 = f"""
SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
"""
print(run_query(q2))

print("\n--- Fact 3: Availability Filtered with IS TRUE ---")
q3 = f"""
SELECT COUNT(*) as total_rows,
       COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_available_rows,
       COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_available_rows,
       ROUND(COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as ga4_available_pct
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
"""
print(run_query(q3))

print("\n==================================================")
print("THREE: FIVE FEATURES FRAME")
print("==================================================")
q_features = f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    SUM(f.gsc_impressions) as impressions_30d,
    SUM(f.gsc_clicks) as clicks_30d,
    AVG(f.gsc_avg_position) as avg_position_30d,
    CASE WHEN SUM(f.ga4_sessions) > 0 THEN SUM(f.ga4_engaged_sessions) * 100.0 / SUM(f.ga4_sessions) ELSE 0.0 END as engagement_rate_30d,
    ANY_VALUE(d.content_age_days) as content_age_at_decision
FROM {TABLES['fact_daily']} f
LEFT JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
WHERE f.report_date BETWEEN '2026-03-01' AND '2026-03-31'
  AND f.ga4_data_available IS TRUE
GROUP BY 1, 2
LIMIT 5
"""
features_df = run_query(q_features)
print(features_df.to_string())

print("\nAvailability Moment explanations:")
print("1. impressions_30d: knowable at the decision moment because it measures search impressions accumulated entirely in the past (March 2026) ending on March 31.")
print("2. clicks_30d: knowable at the decision moment because it counts click events recorded in the past month (March 2026).")
print("3. avg_position_30d: knowable at the decision moment because it summarizes average search ranks during March 2026.")
print("4. engagement_rate_30d: knowable at the decision moment because it is computed from GA4 sessions and engaged sessions recorded in the past month (March 2026).")
print("5. content_age_at_decision: knowable at the decision moment because creation dates are static page metadata established in the past.")

print("\n==================================================")
print("FOUR: LEAKAGE EXPERIMENT")
print("==================================================")

q_leakage = f"""
WITH march_perf AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) as impressions_march,
        SUM(gsc_clicks) as clicks_march,
        AVG(gsc_avg_position) as avg_position_march,
        CASE WHEN SUM(ga4_sessions) > 0 THEN SUM(ga4_engaged_sessions) * 100.0 / SUM(ga4_sessions) ELSE 0.0 END as engagement_rate_march,
        ANY_VALUE(ga4_data_available) as ga4_data_available
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY 1, 2
),
april_perf AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) as impressions_april
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions_march,
    m.clicks_march,
    m.avg_position_march,
    m.engagement_rate_march,
    COALESCE(a.impressions_april, 0) as impressions_april,
    d.content_age_days as content_age_at_decision
FROM march_perf m
LEFT JOIN april_perf a ON m.client_hash_id = a.client_hash_id AND m.content_hash_id = a.content_hash_id
LEFT JOIN {TABLES['dim_content']} d ON m.content_hash_id = d.content_hash_id
WHERE m.ga4_data_available IS TRUE
  AND m.impressions_march >= 10
"""
df_model = run_query(q_leakage)
df_model['is_declining'] = (df_model['impressions_april'] < 0.8 * df_model['impressions_march']).astype(int)
df_model['impressions_decline_ratio'] = df_model['impressions_april'] / (df_model['impressions_march'] + 1e-5)

features_honest = ['impressions_march', 'clicks_march', 'avg_position_march', 'engagement_rate_march', 'content_age_at_decision']
features_leaked = features_honest + ['impressions_decline_ratio']
y = df_model['is_declining']

X_tr_h, X_te_h, y_tr, y_te = train_test_split(df_model[features_honest].fillna(0), y, test_size=0.25, random_state=42, stratify=y)
X_tr_l, X_te_l, _, _ = train_test_split(df_model[features_leaked].fillna(0), y, test_size=0.25, random_state=42, stratify=y)

print("\n--- Model with Leakage (impressions_decline_ratio included) ---")
clf_leak = RandomForestClassifier(n_estimators=100, random_state=42)
clf_leak.fit(X_tr_l, y_tr)
print(classification_report(y_te, clf_leak.predict(X_te_l), digits=3))

print("\n--- Honest Baseline Model (No leakage) ---")
clf_honest = RandomForestClassifier(n_estimators=100, random_state=42)
clf_honest.fit(X_tr_h, y_tr)
print(classification_report(y_te, clf_honest.predict(X_te_h), digits=3))

print("\nSelf-Check: Deleting the leaked feature column and verifying honest score is retained.")
del df_model['impressions_decline_ratio']


Checking/Downloading dataset files from Hugging Face...


HF access failed (403 Client Error. (Request ID: Root=1-6a564087-53b5fa1748c0c12261eb24de;ab69bee8-ad70-4d21-9a57-8b5b019293e9)

Cannot access gated repo for url https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/dim_clients.parquet.
Access to dataset FlyRank/internship-warehouse is restricted and you are not in the authorized list. Visit https://huggingface.co/datasets/FlyRank/internship-warehouse to ask for access.). Falling back to local mock simulation...
Initializing in-memory mock database for warehouse simulation...
Local mock database initialized successfully!
TWO: VERIFICATION QUERIES (month=2026-03)

--- Fact 1: Grain Test (Should return 0 rows if grain holds) ---
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

--- Fact 2: Row Count & Date Span ---
   total_rows   min_date   max_date
0       93000 2026-03-01 2026-03-31

--- Fact 3: Availability Filtered with IS TRUE ---
   total_rows  ga4_available_rows  gsc_availab

## 4. Data limits

*   **Observational Limits & Absence of Causal Proof**: Because this is observational time-series data, we cannot definitively prove that a decline was caused by page stale-ness or lack of updates rather than shifting search intent, technical crawl errors, or competitor changes.
*   **Unbalanced History Start Dates**: Different clients integrated their GA4 and GSC setups at different calendar times. A simple temporal slice (like March 2026) might have missing history or zero-filled GA4 fields for newly onboarded clients, rather than representing true zero organic traffic.

In [8]:
# Check age distributions to show the unbalanced nature of page histories
print("Content Age distributions by percentiles:")
print(df["content_age_days"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_string())

Content Age distributions by percentiles:
0.10    104.0
0.25    132.0
0.50    236.0
0.75    333.0
0.90    463.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.